In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from matplotlib.patches import Patch
import os

# Charger les données
stimuli_table = pd.read_csv(r'Stimuli_information.csv')
participant_data = pd.read_csv(r'PAQ_results.csv')

# Définir les niveaux de bruit
noise_levels = [55, 65, 72]

# Définir les couleurs pour chaque condition de contrôle
control_colors = {
    'No Control': 'gold',              # Gold pour No Control
    'Traditional Algorithm': 'green',  # Vert pour Traditional Algorithm
    'New Algorithm': 'red'             # Rouge pour New Algorithm
}

# Créer le dossier de sortie s'il n'existe pas
output_folder = 'Figures'
os.makedirs(output_folder, exist_ok=True)

for noise_level in noise_levels:
    # Créer un JointGrid pour chaque niveau de bruit
    g = sns.JointGrid(xlim=[-1.5, 1.5], ylim=[-1.5, 1.5], height=10, ratio=5)
    g.ax_joint.set_aspect('equal')
    
    # Stocker les coordonnées X et Y pour chaque condition
    all_x = {'No Control': [], 'Traditional Algorithm': [], 'New Algorithm': []}
    all_y = {'No Control': [], 'Traditional Algorithm': [], 'New Algorithm': []}
    noise_condition = stimuli_table[stimuli_table['Noise_Level'] == noise_level]
    
    for control_condition in control_colors.keys():
        condition_stimuli = noise_condition[noise_condition['Control_Condition'] == control_condition]
        for j in range(condition_stimuli.shape[0]):
            stimulus_name = condition_stimuli['Stimuli'].iloc[j]
            stimulus_idx = np.where(stimuli_table['Stimuli'] == stimulus_name)[0][0]
            x = participant_data.iloc[:, 2 * stimulus_idx + 1].to_numpy()
            y = participant_data.iloc[:, 2 * stimulus_idx + 2].to_numpy()
            valid_indices = np.isfinite(x) & np.isfinite(y)
            all_x[control_condition].extend(x[valid_indices])
            all_y[control_condition].extend(y[valid_indices])
    
    # Tracer les points avec une opacité réduite (pour mettre en avant les zones)
    for control_condition in control_colors.keys():
        g.ax_joint.scatter(all_x[control_condition], all_y[control_condition],
                           label=control_condition, alpha=0.2, s=10,
                           color=control_colors[control_condition], zorder=5)
    
    # Densités marginales sur l'axe X
    # Tracer d'abord Traditional (green) puis No Control (gold)
    sns.kdeplot(x=all_x['Traditional Algorithm'], ax=g.ax_marg_x, fill=True,
                color=control_colors['Traditional Algorithm'], alpha=0.3, zorder=1)
    sns.kdeplot(x=all_x['Traditional Algorithm'], ax=g.ax_marg_x,
                color=control_colors['Traditional Algorithm'], alpha=1, lw=2, zorder=1)
    
    sns.kdeplot(x=all_x['No Control'], ax=g.ax_marg_x, fill=True,
                color=control_colors['No Control'], alpha=0.3, zorder=2)
    sns.kdeplot(x=all_x['No Control'], ax=g.ax_marg_x,
                color=control_colors['No Control'], alpha=1, lw=2, zorder=2)
    
    # Enfin, tracer New Algorithm (rouge) pour qu'il apparaisse devant
    sns.kdeplot(x=all_x['New Algorithm'], ax=g.ax_marg_x, fill=True,
                color=control_colors['New Algorithm'], alpha=0.3, zorder=3)
    sns.kdeplot(x=all_x['New Algorithm'], ax=g.ax_marg_x,
                color=control_colors['New Algorithm'], alpha=1, lw=2, zorder=3)
    
    # Densités marginales sur l'axe Y (mêmes ordres)
    sns.kdeplot(y=all_y['Traditional Algorithm'], ax=g.ax_marg_y, fill=True,
                color=control_colors['Traditional Algorithm'], alpha=0.3, zorder=1)
    sns.kdeplot(y=all_y['Traditional Algorithm'], ax=g.ax_marg_y,
                color=control_colors['Traditional Algorithm'], alpha=1, lw=2, zorder=1)
    
    sns.kdeplot(y=all_y['No Control'], ax=g.ax_marg_y, fill=True,
                color=control_colors['No Control'], alpha=0.3, zorder=2)
    sns.kdeplot(y=all_y['No Control'], ax=g.ax_marg_y,
                color=control_colors['No Control'], alpha=1, lw=2, zorder=2)
    
    sns.kdeplot(y=all_y['New Algorithm'], ax=g.ax_marg_y, fill=True,
                color=control_colors['New Algorithm'], alpha=0.3, zorder=3)
    sns.kdeplot(y=all_y['New Algorithm'], ax=g.ax_marg_y,
                color=control_colors['New Algorithm'], alpha=1, lw=2, zorder=3)
    
    # Sur le plot principal, tracer les zones de densité par condition
    # Tracer d'abord Traditional et No Control, puis New Algorithm (rouge) par-dessus
    sns.kdeplot(x=all_x['Traditional Algorithm'], y=all_y['Traditional Algorithm'], ax=g.ax_joint,
                color=control_colors['Traditional Algorithm'], fill=True, alpha=0.6,
                levels=10, thresh=0.05,
                cmap=sns.light_palette(control_colors['Traditional Algorithm'], as_cmap=True),
                zorder=2)
    
    sns.kdeplot(x=all_x['No Control'], y=all_y['No Control'], ax=g.ax_joint,
                color=control_colors['No Control'], fill=True, alpha=0.6,
                levels=10, thresh=0.05,
                cmap=sns.light_palette(control_colors['No Control'], as_cmap=True),
                zorder=3)
    
    sns.kdeplot(x=all_x['New Algorithm'], y=all_y['New Algorithm'], ax=g.ax_joint,
                color=control_colors['New Algorithm'], fill=True, alpha=0.6,
                levels=10, thresh=0.05,
                cmap=sns.light_palette(control_colors['New Algorithm'], as_cmap=True),
                zorder=4)
    
    # Configurer les axes et le titre
    g.set_axis_labels("Pleasantness (P)", "Eventfulness (E)")
    g.ax_joint.grid(True)
    g.ax_joint.axhline(0, color='k', linestyle='--', linewidth=1.0)
    g.ax_joint.axvline(0, color='k', linestyle='--', linewidth=1.0)
    #g.fig.suptitle(f'Noise Level: {noise_level} dB(A)', y=1.03, fontsize=16)
    
    # Légende
    legend_elements = [
        Patch(facecolor='gold', label='No Control'),
        Patch(facecolor='green', label='NLMS'),
        Patch(facecolor='red', label='SFANC-NLMS')
    ]
    g.fig.legend(handles=legend_elements, loc='upper left', bbox_to_anchor=(1.05, 1),
                 fontsize=12, borderpad=1, handlelength=2, frameon=False)
    plt.subplots_adjust(right=0.85)
    
    # Sauvegarder et afficher la figure
    output_filename = f'{output_folder}/{noise_level}dB_map.pdf'
    plt.savefig(output_filename, format='pdf', bbox_inches='tight')
    plt.show()

print("All figures saved successfully in the 'Figures' folder.")
